# 🏥 Fine-Tuning Expérimental de Mistral 7B (Projet Médical R&D)

Ce notebook Jupyter est conçu pour être exécuté dans **Google Colab (avec accélérateur GPU T4, L4 ou A100)**. Il permet de réaliser le fine-tuning QLoRA (4-bit) du modèle **Mistral-7B-Instruct-v0.3** sur le dataset médical de Hugging Face `ruslanmv/ai-medical-chatbot`.

## 🛠️ Étape 1 : Installation des dépendances optimisées

In [ ]:
# Installation silencieuse des dépendances IA requises pour le QLoRA
!pip install -q transformers bitsandbytes peft accelerate datasets

## 🧬 Étape 2 : Configuration du script d'entraînement pour Mistral

Nous chargeons les modules nécessaires de Hugging Face (PEFT, BitsAndBytes) et configurons l'entraînement. 

> **Note de Sécurité (CYBER)** : Ce script cible explicitement les modules d'attention de Mistral (`q_proj`, `k_proj`, `v_proj`) et utilise le format d'instruction officiel de Mistral (`[INST]`) pour éviter les plantages ou les hallucinations du modèle.

In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset

model_name = "mistralai/Mistral-7B-Instruct-v0.3"
dataset_name = "ruslanmv/ai-medical-chatbot"

print("🔧 Configuration du Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("📦 Chargement du modèle de base avec quantification 4-bit (QLoRA)...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

# Préparation pour l'entraînement QLoRA
model = prepare_model_for_kbit_training(model)

# Configuration LoRA adaptée à Mistral 7B (cibles d'attention spécifiques)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
print(f"✅ Modèle prêt ! Paramètres entraînables : {model.num_parameters()}")

## 📊 Étape 3 : Chargement et Formatage du Dataset Médical

Nous téléchargeons le dataset `ruslanmv/ai-medical-chatbot` et le formatons au format d'instruction Mistral officiel :
`<s>[INST] Requête du Patient [/INST] Réponse du Docteur </s>`

In [ ]:
print("📥 Téléchargement du dataset médical...")
raw_dataset = load_dataset(dataset_name, split="train")

# Utiliser un échantillon de 5000 conversations pour un entraînement rapide sur Colab
subset_size = min(5000, len(raw_dataset))
dataset_subset = raw_dataset.select(range(subset_size))

def format_mistral_prompt(example):
    patient_query = example.get("Patient", "").strip()
    doctor_response = example.get("Doctor", "").strip()
    formatted_text = f"<s>[INST] {patient_query} [/INST] {doctor_response} </s>"
    return {"text": formatted_text}

print("🎨 Formatage du texte...")
formatted_dataset = dataset_subset.map(format_mistral_prompt, remove_columns=dataset_subset.column_names)

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

print("🔧 Tokenisation en cours...")
tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print("✅ Dataset prêt pour l'entraînement !")

## 🚀 Étape 4 : Lancement du Fine-Tuning LoRA

In [ ]:
output_dir = "./medical_mistral_lora"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_drop_last=True,
    fp16=True,
    logging_dir="./logs"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("⏳ Lancement de la boucle d'entraînement (environ 30 à 45 minutes sur GPU T4)...")
trainer.train()

# Sauvegarde finale des poids de l'adaptateur
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"🎉 Entraînement réussi ! L'adaptateur LoRA médical est sauvegardé dans {output_dir}")

## 🧪 Étape 5 : Test interactif du modèle médical

In [ ]:
def ask_medical_bot(query):
    model.eval()
    formatted_input = f"<s>[INST] {query} [/INST]"
    inputs = tokenizer(formatted_input, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

# Test clinique
query = "I have a persistent headache and feel dizzy. What should I do?"
print(f"👤 Patient : {query}")
print(f"🩺 Docteur IA : {ask_medical_bot(query)}")